# IBL session selection — configurable, multi-region pipeline

Session discovery, validation, and ranking using the original analysis criteria:

- Brainwide ephysChoiceWorld recordings containing `LGd`
- male mice, age ≥ 100 days
- at least 400 trials and both choices
- valid stimulus-to-movement latency: 80 ms to 2 s
- complete `prior × |contrast|` grid
- final histology and at least 5 good units in every target region
- usable wheel signal and at least two network regions with ≥ 5 good units
- equal-weight ranking of target-region quality, network coverage, and behavior

For multiple target regions, the best valid probe is selected independently for each
region. Copy the chosen EIDs into the processing notebook.


## 0. Environment


In [ ]:
%pip install -q -U ONE-api ibllib scikit-learn


In [ ]:
import os

os.environ.setdefault('ONE_HTTP_DL_THREADS', '1')


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Iterable, Literal

import numpy as np
import pandas as pd
from IPython.display import display

from one.api import ONE
from brainbox.io.one import SessionLoader, SpikeSortingLoader
from iblatlas.regions import BrainRegions


ONE_PASSWORD = "international"

ONE.setup(base_url='https://openalyx.internationalbrainlab.org', silent=True)
one = ONE(password=ONE_PASSWORD)
print("Connected:", not one.offline)


## 1. User configuration

This is the only cell that normally needs editing. Region names are interpreted using
`ATLAS_MAPPING` for neural counts. `TARGET_REGION_MODE="all"` requires every requested
region to pass the unit threshold; use `"any"` to accept at least one.


In [ ]:
@dataclass(frozen=True)
class SelectionConfig:
    # Search scope
    target_regions: tuple[str, ...]
    atlas_mapping: str = "Beryl"
    target_region_mode: Literal["all", "any"] = "all"
    project: str = "brainwide"
    task_protocol: str = "ephysChoiceWorld"
    required_dataset: str = "spikes.times.npy"

    # Subject filters (None disables a filter)
    sex: str | None = "M"
    min_age_days: int | None = 100

    # Behavioral filters
    min_session_trials: int = 400
    min_valid_trial_fraction: float = 0.0
    require_both_choices: bool = True
    movement_latency_s: tuple[float, float] = (0.080, 2.000)
    expected_priors: tuple[float, ...] = (0.2, 0.5, 0.8)
    expected_contrasts: tuple[float, ...] = (0.0, 0.0625, 0.125, 0.25, 1.0)
    require_complete_condition_grid: bool = True

    # Neural/network filters
    accepted_histology: tuple[str, ...] = ("alf", "resolved")
    min_good_units_per_target_region: int = 5
    min_good_units_per_network_region: int = 5
    min_network_regions: int = 2

    # Wheel/hesitation availability
    evaluate_wheel: bool = True
    require_usable_wheel: bool = True
    low_contrast_max: float = 0.125

    # Rank weights: target population, network coverage, behavior
    rank_weights: tuple[float, float, float] = (1.0, 1.0, 1.0)

    # Batch behavior
    continue_on_error: bool = False


# ------------------------------------------------------------------
# EDIT HERE
# ------------------------------------------------------------------
TARGET_REGIONS = ["LGd"]
# Example: TARGET_REGIONS = ["LGd", "VISa", "CA1"]

SEX = "M"                 # use None for all sexes
MIN_AGE_DAYS = 100         # use None to disable the age filter
MIN_SESSION_TRIALS = 400
MIN_GOOD_UNITS = 5         # required in each target region
REQUIRE_USABLE_WHEEL = True

CONFIG = SelectionConfig(
    target_regions=tuple(TARGET_REGIONS),
    sex=SEX,
    min_age_days=MIN_AGE_DAYS,
    min_session_trials=MIN_SESSION_TRIALS,
    min_good_units_per_target_region=MIN_GOOD_UNITS,
    require_usable_wheel=REQUIRE_USABLE_WHEEL,
)

CONFIG


## 2. Shared validation and trial helpers


In [ ]:
REQUIRED_TRIAL_FIELDS = {
    "contrastLeft",
    "contrastRight",
    "probabilityLeft",
    "choice",
    "stimOn_times",
    "firstMovement_times",
}


def validate_selection_config(config: SelectionConfig) -> None:
    if not config.target_regions:
        raise ValueError("target_regions cannot be empty.")
    if len(set(config.target_regions)) != len(config.target_regions):
        raise ValueError("target_regions contains duplicates.")
    if config.target_region_mode not in {"all", "any"}:
        raise ValueError("target_region_mode must be 'all' or 'any'.")
    low, high = config.movement_latency_s
    if not (0 <= low < high):
        raise ValueError("movement_latency_s must satisfy 0 <= low < high.")
    if config.min_session_trials < 1:
        raise ValueError("min_session_trials must be positive.")
    if not (0 <= config.min_valid_trial_fraction <= 1):
        raise ValueError("min_valid_trial_fraction must be in [0, 1].")
    if config.min_good_units_per_target_region < 1:
        raise ValueError("min_good_units_per_target_region must be positive.")
    if config.min_good_units_per_network_region < 1:
        raise ValueError("min_good_units_per_network_region must be positive.")
    if not config.expected_priors:
        raise ValueError("expected_priors cannot be empty.")
    if not config.expected_contrasts:
        raise ValueError("expected_contrasts cannot be empty.")
    if len(config.rank_weights) != 3 or any(weight < 0 for weight in config.rank_weights):
        raise ValueError("rank_weights must contain three non-negative values.")
    if sum(config.rank_weights) == 0:
        raise ValueError("At least one rank weight must be positive.")


def canonicalize_regions(
    brain_regions: BrainRegions,
    regions: Iterable[str],
    mapping: str,
) -> tuple[str, ...]:
    requested = np.asarray(list(regions), dtype=str)
    mapped = np.asarray(
        brain_regions.acronym2acronym(requested, mapping=mapping),
        dtype=str,
    )
    invalid = requested[np.isin(mapped, ["root", "void", "nan", "None", ""])]
    if len(invalid):
        raise ValueError(
            f"Regions could not be mapped with {mapping}: {invalid.tolist()}"
        )
    if len(set(mapped.tolist())) != len(mapped):
        raise ValueError(
            "Two requested regions collapse to the same mapped region: "
            f"{dict(zip(requested.tolist(), mapped.tolist()))}"
        )
    return tuple(mapped.tolist())


def load_trial_arrays(one: ONE, eid: str) -> dict[str, np.ndarray]:
    trials = one.load_object(eid, "trials", collection="alf")
    missing = REQUIRED_TRIAL_FIELDS - set(trials.keys())
    if missing:
        raise ValueError(f"Missing trial fields: {sorted(missing)}")
    return {
        name: np.asarray(trials[name], dtype=float)
        for name in REQUIRED_TRIAL_FIELDS
    }


def signed_contrast(contrast_left: np.ndarray, contrast_right: np.ndarray) -> np.ndarray:
    # Left stimulus is negative; right stimulus is positive.
    return np.nan_to_num(contrast_right, nan=0.0) - np.nan_to_num(
        contrast_left, nan=0.0
    )


def behavioral_trial_mask(
    arrays: dict[str, np.ndarray],
    latency_bounds: tuple[float, float],
) -> tuple[np.ndarray, np.ndarray]:
    latency = arrays["firstMovement_times"] - arrays["stimOn_times"]
    low, high = latency_bounds
    mask = (
        np.isfinite(arrays["probabilityLeft"])
        & np.isfinite(arrays["choice"])
        & np.isfinite(arrays["stimOn_times"])
        & np.isfinite(arrays["firstMovement_times"])
        & (latency >= low)
        & (latency <= high)
    )
    return mask, latency


def error_row(stage: str, eid: str | None, subject: str | None, exc: Exception) -> dict:
    return {
        "stage": stage,
        "eid": eid,
        "subject": subject,
        "error_type": type(exc).__name__,
        "message": str(exc),
    }


## 3. Candidate discovery and subject metadata


In [ ]:
def search_candidate_insertions(one: ONE, config: SelectionConfig) -> pd.DataFrame:
    # Union insertion searches for all requested target acronyms.
    rows: list[dict[str, Any]] = []

    for search_region in config.target_regions:
        pids, insertions = one.search_insertions(
            atlas_acronym=search_region,
            datasets=config.required_dataset,
            project=config.project,
            task_protocol=config.task_protocol,
            query_type="remote",
            details=True,
        )

        for pid, insertion in zip(pids, insertions):
            session = insertion["session_info"]
            rows.append(
                {
                    "subject": session["subject"],
                    "eid": str(insertion["session"]),
                    "pid": str(pid),
                    "probe_name": insertion["name"],
                    "start_time": session["start_time"],
                    "lab": session["lab"],
                    "task_protocol": session["task_protocol"],
                    "matched_search_region": search_region,
                }
            )

    if not rows:
        raise ValueError(
            "No insertions matched the search. Check target acronyms and search scope."
        )

    return (
        pd.DataFrame(rows)
        .drop_duplicates(subset=["pid", "matched_search_region"])
        .sort_values(["subject", "start_time", "probe_name"])
        .reset_index(drop=True)
    )


def fetch_subject_metadata(one: ONE, subjects: Iterable[str]) -> pd.DataFrame:
    rows = []
    for subject in sorted(set(subjects)):
        records = one.alyx.rest("subjects", "list", nickname=subject)
        exact = [record for record in records if record.get("nickname") == subject]
        if len(exact) != 1:
            raise ValueError(
                f"{subject}: expected one exact subject record, found {len(exact)}."
            )
        record = exact[0]
        rows.append(
            {
                "subject": subject,
                "sex": record.get("sex"),
                "birth_date": record.get("birth_date"),
            }
        )
    return pd.DataFrame(rows)


def build_candidate_sessions(
    one: ONE,
    insertion_df: pd.DataFrame,
    config: SelectionConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    subject_df = fetch_subject_metadata(one, insertion_df["subject"])
    recording_df = insertion_df.merge(subject_df, on="subject", how="left")

    recording_df["session_date"] = pd.to_datetime(
        recording_df["start_time"].astype(str).str[:10],
        format="%Y-%m-%d",
        errors="coerce",
    )
    recording_df["birth_date"] = pd.to_datetime(
        recording_df["birth_date"], format="%Y-%m-%d", errors="coerce"
    )
    recording_df["age_days"] = (
        recording_df["session_date"] - recording_df["birth_date"]
    ).dt.days

    keep = np.ones(len(recording_df), dtype=bool)
    if config.sex is not None:
        keep &= recording_df["sex"].eq(config.sex).to_numpy()
    if config.min_age_days is not None:
        keep &= recording_df["age_days"].ge(config.min_age_days).to_numpy()

    filtered_recordings = recording_df.loc[keep].copy().reset_index(drop=True)
    sessions = (
        filtered_recordings[
            ["eid", "subject", "session_date", "age_days", "sex", "lab"]
        ]
        .drop_duplicates(subset="eid")
        .sort_values(["subject", "session_date"])
        .reset_index(drop=True)
    )
    return sessions, filtered_recordings


## 4. Behavioral validity and prior × contrast coverage


In [ ]:
def evaluate_behavior(
    one: ONE,
    sessions: pd.DataFrame,
    config: SelectionConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    summaries: list[dict] = []
    condition_tables: list[pd.DataFrame] = []
    errors: list[dict] = []

    for session in sessions.itertuples(index=False):
        try:
            arrays = load_trial_arrays(one, session.eid)
            valid, _ = behavioral_trial_mask(arrays, config.movement_latency_s)
            signed = signed_contrast(
                arrays["contrastLeft"], arrays["contrastRight"]
            )
            n_trials = len(arrays["choice"])
            n_valid = int(valid.sum())

            summaries.append(
                {
                    "eid": session.eid,
                    "subject": session.subject,
                    "n_trials": n_trials,
                    "n_valid_trials": n_valid,
                    "valid_trial_fraction": n_valid / n_trials if n_trials else np.nan,
                    "priors": sorted(
                        np.unique(arrays["probabilityLeft"][valid]).tolist()
                    ),
                    "choices": sorted(np.unique(arrays["choice"][valid]).tolist()),
                }
            )

            counts = (
                pd.DataFrame(
                    {
                        "probability_left": arrays["probabilityLeft"][valid],
                        "signed_contrast": signed[valid],
                    }
                )
                .groupby(["probability_left", "signed_contrast"], as_index=False)
                .size()
                .rename(columns={"size": "n_trials"})
            )
            counts["eid"] = session.eid
            counts["subject"] = session.subject
            condition_tables.append(counts)
        except Exception as exc:
            errors.append(error_row("behavior", session.eid, session.subject, exc))
            if not config.continue_on_error:
                raise

    summary_df = pd.DataFrame(
        summaries,
        columns=[
            "eid",
            "subject",
            "n_trials",
            "n_valid_trials",
            "valid_trial_fraction",
            "priors",
            "choices",
        ],
    )
    condition_df = (
        pd.concat(condition_tables, ignore_index=True)
        if condition_tables
        else pd.DataFrame(
            columns=[
                "probability_left",
                "signed_contrast",
                "n_trials",
                "eid",
                "subject",
            ]
        )
    )

    grid_rows: list[dict] = []
    absolute_counts = condition_df.copy()
    if not absolute_counts.empty:
        absolute_counts["contrast"] = absolute_counts["signed_contrast"].abs()
        absolute_counts = (
            absolute_counts.groupby(
                ["eid", "subject", "probability_left", "contrast"],
                as_index=False,
            )["n_trials"]
            .sum()
        )

    for session in summary_df.itertuples(index=False):
        current = absolute_counts[absolute_counts["eid"].eq(session.eid)]
        required_counts: list[int] = []
        for prior in config.expected_priors:
            for contrast in config.expected_contrasts:
                match = current[
                    np.isclose(current["probability_left"], prior)
                    & np.isclose(current["contrast"], contrast)
                ]
                required_counts.append(int(match["n_trials"].sum()) if len(match) else 0)

        counts_array = np.asarray(required_counts, dtype=int)
        grid_rows.append(
            {
                "eid": session.eid,
                "subject": session.subject,
                "n_required_conditions_present": int((counts_array > 0).sum()),
                "min_trials_required_condition": int(counts_array.min()),
                "median_trials_required_condition": float(np.median(counts_array)),
                "complete_prior_contrast_grid": bool(np.all(counts_array > 0)),
            }
        )

    grid_df = pd.DataFrame(
        grid_rows,
        columns=[
            "eid",
            "subject",
            "n_required_conditions_present",
            "min_trials_required_condition",
            "median_trials_required_condition",
            "complete_prior_contrast_grid",
        ],
    )
    behavior_df = summary_df.merge(grid_df, on=["eid", "subject"], how="inner")

    if not behavior_df.empty:
        keep = (
            behavior_df["n_trials"].ge(config.min_session_trials)
            & behavior_df["valid_trial_fraction"].ge(config.min_valid_trial_fraction)
        )
        if config.require_both_choices:
            keep &= behavior_df["choices"].apply(
                lambda values: -1.0 in values and 1.0 in values
            )
        if config.require_complete_condition_grid:
            keep &= behavior_df["complete_prior_contrast_grid"]
        behavior_df = behavior_df.loc[keep].copy().reset_index(drop=True)

    return behavior_df, summary_df, condition_df, pd.DataFrame(errors)


## 5. Neural metadata cache and session-wide region coverage

Cluster metadata are downloaded once per `EID × probe`. Only QC label 1 units and
probes with accepted histology contribute to target and network counts.


In [ ]:
class ClusterMetadataCache:
    def __init__(self) -> None:
        self._cache: dict[tuple[str, str], dict[str, Any]] = {}

    def load(
        self,
        one: ONE,
        brain_regions: BrainRegions,
        eid: str,
        probe_name: str,
    ) -> dict[str, Any]:
        key = (eid, probe_name)
        if key in self._cache:
            return self._cache[key]

        loader = SpikeSortingLoader(eid=eid, pname=probe_name, one=one)
        clusters = loader.load_spike_sorting_object("clusters")
        channels = loader.load_channels()

        if "metrics" not in clusters:
            raise ValueError("clusters.metrics not found.")
        metrics = pd.DataFrame(clusters["metrics"])
        if "label" not in metrics:
            raise ValueError("Cluster QC label not found in clusters.metrics.")

        raw_channel_index = np.asarray(clusters["channels"], dtype=float)
        if len(metrics) != len(raw_channel_index):
            raise ValueError("Cluster metrics and channel indices have different lengths.")

        n_channels = len(channels["acronym"])
        valid_channel = (
            np.isfinite(raw_channel_index)
            & (raw_channel_index >= 0)
            & (raw_channel_index < n_channels)
        )
        safe_index = np.where(valid_channel, raw_channel_index, 0).astype(int)

        allen = np.full(len(raw_channel_index), "void", dtype=object)
        x = np.full(len(raw_channel_index), np.nan, dtype=float)
        allen[valid_channel] = np.asarray(channels["acronym"])[safe_index[valid_channel]]
        x[valid_channel] = np.asarray(channels["x"], dtype=float)[safe_index[valid_channel]]

        cluster_df = pd.DataFrame(
            {
                "cluster_index": np.arange(len(raw_channel_index)),
                "is_good": metrics["label"].to_numpy() == 1,
                "acronym_allen": allen,
                "acronym_beryl": brain_regions.acronym2acronym(
                    allen, mapping="Beryl"
                ),
                "acronym_cosmos": brain_regions.acronym2acronym(
                    allen, mapping="Cosmos"
                ),
                "x": x,
            }
        )

        result = {"clusters": cluster_df, "histology": loader.histology}
        self._cache[key] = result
        return result


def hemisphere_from_x(x: np.ndarray) -> str | None:
    values = np.asarray(x, dtype=float)
    if not len(values):
        return None
    if np.all(values < 0):
        return "left"
    if np.all(values > 0):
        return "right"
    return "mixed"


def evaluate_neural_coverage(
    one: ONE,
    sessions: pd.DataFrame,
    config: SelectionConfig,
    mapped_target_regions: tuple[str, ...],
    brain_regions: BrainRegions,
    cache: ClusterMetadataCache,
) -> dict[str, pd.DataFrame]:
    summaries: list[dict] = []
    beryl_rows: list[dict] = []
    cosmos_rows: list[dict] = []
    target_rows: list[dict] = []
    probe_rows: list[dict] = []
    errors: list[dict] = []

    for session in sessions.itertuples(index=False):
        try:
            insertions = one.alyx.rest("insertions", "list", session=session.eid)
            good_tables: list[pd.DataFrame] = []
            n_final_histology = 0

            for insertion in insertions:
                probe_name = insertion["name"]
                pid = str(insertion.get("id", ""))
                try:
                    neural = cache.load(
                        one, brain_regions, session.eid, probe_name
                    )
                    histology = neural["histology"]
                    probe_rows.append(
                        {
                            "eid": session.eid,
                            "subject": session.subject,
                            "pid": pid,
                            "probe_name": probe_name,
                            "histology": histology,
                            "histology_accepted": histology
                            in config.accepted_histology,
                            "n_units": len(neural["clusters"]),
                            "n_good_units": int(
                                neural["clusters"]["is_good"].sum()
                            ),
                        }
                    )
                    if histology not in config.accepted_histology:
                        continue
                    n_final_histology += 1
                    good = neural["clusters"].loc[
                        neural["clusters"]["is_good"]
                    ].copy()
                    good["pid"] = pid
                    good["probe_name"] = probe_name
                    good_tables.append(good)
                except Exception as probe_exc:
                    errors.append(
                        error_row(
                            "neural_probe",
                            session.eid,
                            session.subject,
                            RuntimeError(f"{probe_name}: {probe_exc}"),
                        )
                    )
                    if not config.continue_on_error:
                        raise

            if not good_tables:
                raise ValueError("No accepted-histology probe with good units.")

            session_good = pd.concat(good_tables, ignore_index=True)
            session_good["acronym_target"] = brain_regions.acronym2acronym(
                session_good["acronym_allen"].to_numpy(),
                mapping=config.atlas_mapping,
            )
            beryl_counts = (
                session_good.loc[
                    ~session_good["acronym_beryl"].isin(["root", "void"])
                ]
                .groupby("acronym_beryl")
                .size()
                .sort_values(ascending=False)
            )
            cosmos_counts = (
                session_good.loc[
                    ~session_good["acronym_cosmos"].isin(["root", "void"])
                ]
                .groupby("acronym_cosmos")
                .size()
                .sort_values(ascending=False)
            )

            for region, n_units in beryl_counts.items():
                beryl_rows.append(
                    {
                        "eid": session.eid,
                        "subject": session.subject,
                        "region": region,
                        "n_good_units": int(n_units),
                    }
                )
            for division, n_units in cosmos_counts.items():
                cosmos_rows.append(
                    {
                        "eid": session.eid,
                        "subject": session.subject,
                        "division": division,
                        "n_good_units": int(n_units),
                    }
                )

            target_counts: list[int] = []
            target_passes: list[bool] = []
            for requested, mapped in zip(
                config.target_regions, mapped_target_regions
            ):
                all_target_units = session_good[
                    session_good["acronym_target"].eq(mapped)
                ]
                probe_counts = (
                    all_target_units.groupby(
                        ["pid", "probe_name"], as_index=False
                    )
                    .size()
                    .rename(columns={"size": "n_good_units"})
                    .sort_values(
                        ["n_good_units", "probe_name", "pid"],
                        ascending=[False, True, True],
                    )
                    .reset_index(drop=True)
                )

                if probe_counts.empty:
                    selected_pid = None
                    selected_probe = None
                    n_units = 0
                    hemisphere = None
                else:
                    best_probe = probe_counts.iloc[0]
                    selected_pid = best_probe["pid"]
                    selected_probe = best_probe["probe_name"]
                    n_units = int(best_probe["n_good_units"])
                    selected_units = all_target_units[
                        all_target_units["pid"].eq(selected_pid)
                        & all_target_units["probe_name"].eq(selected_probe)
                    ]
                    hemisphere = hemisphere_from_x(
                        selected_units["x"].to_numpy(dtype=float)
                    )

                passed = n_units >= config.min_good_units_per_target_region
                target_counts.append(n_units)
                target_passes.append(passed)
                target_rows.append(
                    {
                        "eid": session.eid,
                        "subject": session.subject,
                        "requested_region": requested,
                        "mapped_region": mapped,
                        "pid": selected_pid,
                        "probe_name": selected_probe,
                        "n_good_units": n_units,
                        "hemisphere": hemisphere,
                        "passes_threshold": passed,
                    }
                )

            target_viable = (
                all(target_passes)
                if config.target_region_mode == "all"
                else any(target_passes)
            )
            summaries.append(
                {
                    "eid": session.eid,
                    "subject": session.subject,
                    "n_recorded_probes": len(insertions),
                    "n_final_histology_probes": n_final_histology,
                    "n_good_units_session": len(session_good),
                    "n_regions_ge_threshold": int(
                        (beryl_counts >= config.min_good_units_per_network_region).sum()
                    ),
                    "n_cosmos_divisions_ge_threshold": int(
                        (cosmos_counts >= config.min_good_units_per_network_region).sum()
                    ),
                    "target_min_good_units": int(min(target_counts)),
                    "target_total_good_units": int(sum(target_counts)),
                    "n_target_regions_passing": int(sum(target_passes)),
                    "target_regions_viable": bool(target_viable),
                }
            )
        except Exception as exc:
            errors.append(error_row("neural_session", session.eid, session.subject, exc))
            if not config.continue_on_error:
                raise

    return {
        "summary": pd.DataFrame(
            summaries,
            columns=[
                "eid",
                "subject",
                "n_recorded_probes",
                "n_final_histology_probes",
                "n_good_units_session",
                "n_regions_ge_threshold",
                "n_cosmos_divisions_ge_threshold",
                "target_min_good_units",
                "target_total_good_units",
                "n_target_regions_passing",
                "target_regions_viable",
            ],
        ),
        "region_counts": pd.DataFrame(
            beryl_rows,
            columns=["eid", "subject", "region", "n_good_units"],
        ),
        "division_counts": pd.DataFrame(
            cosmos_rows,
            columns=["eid", "subject", "division", "n_good_units"],
        ),
        "target_counts": pd.DataFrame(
            target_rows,
            columns=[
                "eid",
                "subject",
                "requested_region",
                "mapped_region",
                "pid",
                "probe_name",
                "n_good_units",
                "hemisphere",
                "passes_threshold",
            ],
        ),
        "probe_quality": pd.DataFrame(
            probe_rows,
            columns=[
                "eid",
                "subject",
                "pid",
                "probe_name",
                "histology",
                "histology_accepted",
                "n_units",
                "n_good_units",
            ],
        ),
        "errors": pd.DataFrame(
            errors,
            columns=["stage", "eid", "subject", "error_type", "message"],
        ),
    }


## 6. Wheel validation


In [ ]:
def evaluate_wheel(
    one: ONE,
    sessions: pd.DataFrame,
    config: SelectionConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict] = []
    errors: list[dict] = []

    for session in sessions.itertuples(index=False):
        try:
            loader = SessionLoader(eid=session.eid, one=one)
            loader.load_wheel()
            wheel = loader.wheel
            arrays = load_trial_arrays(one, session.eid)
            latency = (
                arrays["firstMovement_times"] - arrays["stimOn_times"]
            )
            latency_low, latency_high = config.movement_latency_s
            valid = (
                np.isfinite(arrays["stimOn_times"])
                & np.isfinite(arrays["firstMovement_times"])
                & (latency >= latency_low)
                & (latency <= latency_high)
            )
            signed = signed_contrast(
                arrays["contrastLeft"], arrays["contrastRight"]
            )
            low_contrast = valid & (np.abs(signed) <= config.low_contrast_max)

            wheel_times = wheel["times"].to_numpy(dtype=float)
            wheel_velocity = wheel["velocity"].to_numpy(dtype=float)
            finite_velocity = np.isfinite(wheel_velocity)
            monotonic = len(wheel_times) > 1 and np.all(np.diff(wheel_times) > 0)

            valid_stim = arrays["stimOn_times"][valid]
            valid_move = arrays["firstMovement_times"][valid]
            covers_trials = (
                len(wheel_times) > 0
                and len(valid_stim) > 0
                and wheel_times[0] <= np.min(valid_stim)
                and wheel_times[-1] >= np.max(valid_move)
            )
            usable = bool(monotonic and covers_trials and finite_velocity.any())

            low_latency = latency[low_contrast]
            if len(low_latency) > 1:
                q25, q75 = np.percentile(low_latency, [25, 75])
                latency_iqr_ms = float((q75 - q25) * 1000)
            else:
                latency_iqr_ms = np.nan

            rows.append(
                {
                    "eid": session.eid,
                    "subject": session.subject,
                    "wheel_samples": len(wheel),
                    "wheel_velocity_finite_fraction": float(finite_velocity.mean()),
                    "wheel_velocity_std": float(np.nanstd(wheel_velocity)),
                    "wheel_abs_velocity_p95": float(
                        np.nanpercentile(np.abs(wheel_velocity), 95)
                    ),
                    "wheel_covers_trials": bool(covers_trials),
                    "wheel_usable": usable,
                    "low_contrast_valid_trials": int(low_contrast.sum()),
                    "low_contrast_latency_iqr_ms": latency_iqr_ms,
                }
            )
        except Exception as exc:
            errors.append(error_row("wheel", session.eid, session.subject, exc))
            rows.append(
                {
                    "eid": session.eid,
                    "subject": session.subject,
                    "wheel_usable": False,
                    "low_contrast_valid_trials": 0,
                    "low_contrast_latency_iqr_ms": np.nan,
                }
            )
            if not config.continue_on_error:
                raise

    return (
        pd.DataFrame(
            rows,
            columns=[
                "eid",
                "subject",
                "wheel_samples",
                "wheel_velocity_finite_fraction",
                "wheel_velocity_std",
                "wheel_abs_velocity_p95",
                "wheel_covers_trials",
                "wheel_usable",
                "low_contrast_valid_trials",
                "low_contrast_latency_iqr_ms",
            ],
        ),
        pd.DataFrame(
            errors,
            columns=["stage", "eid", "subject", "error_type", "message"],
        ),
    )


## 7. Ranking and end-to-end orchestration


In [ ]:
@dataclass
class SelectionResult:
    config: SelectionConfig
    mapped_target_regions: tuple[str, ...]
    ranked_sessions: pd.DataFrame
    target_region_counts: pd.DataFrame
    network_region_counts: pd.DataFrame
    network_division_counts: pd.DataFrame
    probe_quality: pd.DataFrame
    behavioral_summary: pd.DataFrame
    trial_condition_counts: pd.DataFrame
    wheel_quality: pd.DataFrame
    pipeline_counts: pd.DataFrame
    errors: pd.DataFrame


def rank_sessions(sessions: pd.DataFrame, config: SelectionConfig) -> pd.DataFrame:
    ranked = sessions.copy()
    if ranked.empty:
        for column in [
            "target_rank",
            "network_rank",
            "behavior_rank",
            "selection_score",
        ]:
            ranked[column] = pd.Series(dtype=float)
        return ranked

    ranked["target_rank"] = ranked["target_min_good_units"].rank(
        ascending=False, method="min"
    )
    region_rank = ranked["n_regions_ge_threshold"].rank(
        ascending=False, method="min"
    )
    division_rank = ranked["n_cosmos_divisions_ge_threshold"].rank(
        ascending=False, method="min"
    )
    ranked["network_rank"] = (region_rank + division_rank) / 2

    trial_rank = ranked["n_valid_trials"].rank(ascending=False, method="min")
    condition_rank = ranked["min_trials_required_condition"].rank(
        ascending=False, method="min"
    )
    ranked["behavior_rank"] = (trial_rank + condition_rank) / 2

    target_weight, network_weight, behavior_weight = config.rank_weights
    ranked["selection_score"] = (
        target_weight * ranked["target_rank"]
        + network_weight * ranked["network_rank"]
        + behavior_weight * ranked["behavior_rank"]
    )
    return (
        ranked.sort_values(
            [
                "selection_score",
                "low_contrast_valid_trials",
                "target_min_good_units",
            ],
            ascending=[True, False, False],
        )
        .reset_index(drop=True)
    )


def run_session_selection(one: ONE, config: SelectionConfig) -> SelectionResult:
    validate_selection_config(config)
    brain_regions = BrainRegions()
    mapped_targets = canonicalize_regions(
        brain_regions, config.target_regions, config.atlas_mapping
    )

    insertions = search_candidate_insertions(one, config)
    candidate_sessions, _ = build_candidate_sessions(one, insertions, config)
    behavior_valid, behavior_all, condition_counts, behavior_errors = (
        evaluate_behavior(one, candidate_sessions, config)
    )

    cache = ClusterMetadataCache()
    neural = evaluate_neural_coverage(
        one,
        behavior_valid[["eid", "subject"]],
        config,
        mapped_targets,
        brain_regions,
        cache,
    )

    merged = (
        behavior_valid.merge(
            candidate_sessions,
            on=["eid", "subject"],
            how="left",
        )
        .merge(neural["summary"], on=["eid", "subject"], how="inner")
    )
    if not merged.empty:
        merged = merged[
            merged["target_regions_viable"]
            & merged["n_regions_ge_threshold"].ge(config.min_network_regions)
        ].copy()
    target_network_valid_count = len(merged)

    if config.evaluate_wheel:
        wheel_quality, wheel_errors = evaluate_wheel(
            one, merged[["eid", "subject"]], config
        )
        merged = merged.merge(
            wheel_quality, on=["eid", "subject"], how="left"
        )
        if config.require_usable_wheel and not merged.empty:
            merged = merged[merged["wheel_usable"].fillna(False)].copy()
    else:
        wheel_quality = pd.DataFrame(columns=["eid", "subject"])
        wheel_errors = pd.DataFrame()
        merged["wheel_usable"] = np.nan
        merged["low_contrast_valid_trials"] = 0
        merged["low_contrast_latency_iqr_ms"] = np.nan

    ranked = rank_sessions(merged.reset_index(drop=True), config)

    error_frames = [
        frame
        for frame in [behavior_errors, neural["errors"], wheel_errors]
        if not frame.empty
    ]
    errors = (
        pd.concat(error_frames, ignore_index=True)
        if error_frames
        else pd.DataFrame(
            columns=["stage", "eid", "subject", "error_type", "message"]
        )
    )

    pipeline_counts = pd.DataFrame(
        {
            "stage": [
                "matched insertions",
                "subject-filtered sessions",
                "behavior-valid sessions",
                "target/network-valid sessions",
                "wheel-valid sessions",
                "final ranked sessions",
            ],
            "count": [
                len(insertions),
                len(candidate_sessions),
                len(behavior_valid),
                target_network_valid_count,
                len(merged),
                len(ranked),
            ],
        }
    )

    return SelectionResult(
        config=config,
        mapped_target_regions=mapped_targets,
        ranked_sessions=ranked,
        target_region_counts=neural["target_counts"],
        network_region_counts=neural["region_counts"],
        network_division_counts=neural["division_counts"],
        probe_quality=neural["probe_quality"],
        behavioral_summary=behavior_all,
        trial_condition_counts=condition_counts,
        wheel_quality=wheel_quality,
        pipeline_counts=pipeline_counts,
        errors=errors,
    )


## 8. Run selection

This is the only expensive call in the notebook. It downloads behavioral, cluster
metadata, channel anatomy, and wheel datasets as required.


In [ ]:
selection = run_session_selection(one, CONFIG)
ranked_sessions_df = selection.ranked_sessions

display(selection.pipeline_counts)
print("Mapped target regions:", selection.mapped_target_regions)
print("Errors recorded:", len(selection.errors))


In [ ]:
RANKED_COLUMNS = [
    "subject",
    "eid",
    "session_date",
    "age_days",
    "target_min_good_units",
    "target_total_good_units",
    "n_regions_ge_threshold",
    "n_cosmos_divisions_ge_threshold",
    "n_valid_trials",
    "min_trials_required_condition",
    "low_contrast_valid_trials",
    "low_contrast_latency_iqr_ms",
    "target_rank",
    "network_rank",
    "behavior_rank",
    "selection_score",
]

display(ranked_sessions_df[RANKED_COLUMNS])

# Convenience view when the decision is "which mouse?" rather than
# "which recording?": keep the best-ranked session for each subject.
best_session_per_mouse_df = (
    ranked_sessions_df.drop_duplicates(subset="subject", keep="first")
    .reset_index(drop=True)
)


## 9. Inspect a selected session

Change `SELECTED_RANK` to inspect another row. The processing notebook only needs the
EID list and region list; PIDs are discovered automatically.


In [ ]:
if ranked_sessions_df.empty:
    raise ValueError("No session passed all configured filters.")

SELECTED_RANK = 0
selected = ranked_sessions_df.iloc[SELECTED_RANK]
SELECTED_EID = selected["eid"]
SELECTED_SUBJECT = selected["subject"]

print("SELECTED DATASET")
print("=" * 60)
print("Rank:", SELECTED_RANK + 1)
print("Mouse:", SELECTED_SUBJECT)
print("Session EID:", SELECTED_EID)
print("Mapped target regions:", selection.mapped_target_regions)
print("Minimum good units across targets:", int(selected["target_min_good_units"]))
print("Valid trials:", int(selected["n_valid_trials"]))
print("Network regions above threshold:", int(selected["n_regions_ge_threshold"]))

display(
    selection.target_region_counts[
        selection.target_region_counts["eid"].eq(SELECTED_EID)
    ].reset_index(drop=True)
)
display(
    selection.network_region_counts[
        selection.network_region_counts["eid"].eq(SELECTED_EID)
        & selection.network_region_counts["n_good_units"].ge(
            CONFIG.min_good_units_per_network_region
        )
    ]
    .sort_values("n_good_units", ascending=False)
    .reset_index(drop=True)
)


In [ ]:
# Copy this directly into SESSION_IDS in the processing notebook.
TOP_N = 5
SESSION_IDS_FOR_PROCESSING = ranked_sessions_df["eid"].head(TOP_N).tolist()
SESSION_IDS_FOR_PROCESSING


## 10. Diagnostics (optional)

Errors are reported with their stage and EID. The default behavior is fail-fast.


In [ ]:
if selection.errors.empty:
    print("No batch errors were recorded.")
else:
    display(selection.errors)
